In [1]:
"""
Epistemic Uncertainty via DDIM Inversion + Forward Diffusion Ensembles
========================================================================
Kaggle Notebook script — GPU T4 x2 environment.
 
Pipeline:
  1. Load a real, already-preprocessed CXR image (from the .h5 file produced
     by the earlier VinBigData preprocessing pipeline: vindr_cxr_512.h5,
     float32, normalized to [-1, 1], keyed by image_id).
  2. Deterministically invert it into the diffusion model's latent space
     via DDIM Inversion -> exact starting latent z_T.
  3. From that SAME z_T, run N=20 stochastic DDIM reverse passes (eta=1.0)
     to build an ensemble of plausible reconstructions.
  4. Accumulate pixel-wise mean/variance across the ensemble using
     Welford's online algorithm (O(1) memory — never stores all 20 images).
  5. Process in chunks of 50 images, restricted to 2 rare pathology
     classes, splitting each chunk across both T4 GPUs. Save the running
     variance (epistemic uncertainty heatmap) as compressed .npz after
     every chunk.
 
Paste each "# %% [CELL N]" block into its own notebook cell, or run as-is.
"""
 
# %% [CELL 0] --------------------------------------------------------------
# Install / verify dependencies
# ---------------------------------------------------------------------------
import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "diffusers>=0.27.0", "transformers", "accelerate", "safetensors", "huggingface_hub"],
    check=False,
)
 
 
# %% [CELL 0b] -------------------------------------------------------------
# HuggingFace authentication (required for gated stanfordmimi/RoentGen-v2)
# ---------------------------------------------------------------------------
# Preferred: store your token as a Kaggle Secret named "HF_TOKEN"
#   Notebook menu -> Add-ons -> Secrets -> Add a new secret
#     Label: HF_TOKEN   Value: <your hf_... token with read access>
# then re-attach it to this notebook. This avoids ever hardcoding the token
# in the notebook itself (safe to share/publish the notebook afterward).
import os
 
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    HF_TOKEN = user_secrets.get_secret("MS_RoentGen")
    print("HF_TOKEN loaded from Kaggle Secrets.")
except Exception:
    # Fallbacks: an environment variable, or interactive prompt (won't be
    # persisted anywhere, but works if Secrets isn't set up)
    HF_TOKEN = os.environ.get("HF_TOKEN")
    if HF_TOKEN:
        print("HF_TOKEN loaded from environment variable.")
    else:
        try:
            from getpass import getpass
            HF_TOKEN = getpass("Enter your HuggingFace token (input hidden), "
                                "or press Enter to skip and rely on the "
                                "public SD1.5 fallback: ") or None
        except Exception:
            HF_TOKEN = None
 
if HF_TOKEN:
    try:
        from huggingface_hub import login
        login(token=HF_TOKEN)
        print("Logged in to HuggingFace Hub — gated models (RoentGen-v2) accessible.")
    except Exception as e:
        print(f"HuggingFace login failed ({e}); will fall back to public models only.")
        HF_TOKEN = None
else:
    print("No HF_TOKEN found — RoentGen-v2 load attempt will fail and the "
          "pipeline will fall back to runwayml/stable-diffusion-v1-5.")
 

HF_TOKEN loaded from Kaggle Secrets.
Logged in to HuggingFace Hub — gated models (RoentGen-v2) accessible.


In [2]:
# %% [CELL 1] --------------------------------------------------------------
# Imports & global config
# ---------------------------------------------------------------------------
import ast
import gc
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
 
import h5py
import numpy as np
import pandas as pd
import torch
from diffusers import StableDiffusionPipeline, DDIMScheduler, DDIMInverseScheduler
from tqdm.auto import tqdm
 
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
 
# ---- Paths --------------------------------------------------------------
# Reuses the outputs of the earlier VinBigData preprocessing pipeline.
H5_PATH = Path("/kaggle/input/datasets/kartikichandratre/vindr-cxr-512/vindr_cxr_512.h5")
METADATA_CSV = Path("/kaggle/input/datasets/kartikichandratre/vindr-cxr-512/vindr_cxr_train.csv")   # has 'labels' col
OUT_DIR = Path("/kaggle/working")
OUT_DIR.mkdir(parents=True, exist_ok=True)
 
# ---- Model candidates (try gated medical model, fall back to base SD) ---
# ---- Model candidates (prioritize local Kaggle inputs, fall back to HF) ---
LOCAL_CANDIDATES = []
INPUT_DIR = Path("/kaggle/input")

if INPUT_DIR.exists():
    print("Scanning Kaggle Inputs for local model weights...")
    # Recursively find any directory containing 'model_index.json' (the Diffusers signature)
    for p in INPUT_DIR.rglob("model_index.json"):
        local_path = str(p.parent)
        LOCAL_CANDIDATES.append(local_path)
        print(f" -> Found local model at: {local_path}")

MODEL_CANDIDATES = LOCAL_CANDIDATES + [
    "stanfordmimi/RoentGen-v2",
    "runwayml/stable-diffusion-v1-5",
]
print(f"Resolved model search order: {MODEL_CANDIDATES}")
 
# ---- Diffusion / uncertainty params -------------------------------------
NUM_INVERSION_STEPS = 30      # DDIM inversion steps (fewer = faster, less exact)
NUM_INFERENCE_STEPS = 30      # reverse/generation steps per ensemble member
NUM_ENSEMBLE = 20             # N ensemble samples per image (fixed by spec)
ETA = 1                     # stochasticity for ensemble generation (fixed by spec)
GUIDANCE_SCALE = 1.0          # MUST stay 1.0 — see note in get_text_embeddings()
IMG_SIZE = 512
CHUNK_SIZE = 50
 
# ---- Target rare findings ------------------------------------------------
# Inspect actual class frequencies first (run this once, then hardcode the
# two rarest clinically-relevant classes below):
#   annotations["class_name"].value_counts().tail(10)
RARE_FINDINGS = ["Atelectasis"]   # <-- EDIT to match your CSV's classes
 
# ---- Devices --------------------------------------------------------------
NUM_GPUS = torch.cuda.device_count()
DEVICES = [f"cuda:{i}" for i in range(NUM_GPUS)] if NUM_GPUS > 0 else ["cpu"]
print(f"Detected devices: {DEVICES}")

# %% [CELL 2] --------------------------------------------------------------
# Pipeline loading (one full pipeline copy per GPU — required for true
# parallelism across the T4 x2 setup; each GPU processes batch size 1)
# ---------------------------------------------------------------------------
def load_pipeline(device: str) -> StableDiffusionPipeline:
    pipe = None
    for model_id in MODEL_CANDIDATES:
        try:
            # Check if candidate points to a local directory on the Kaggle SSD
            is_local = Path(model_id).exists()
            
            kwargs = {
                "torch_dtype": torch.float16,
                "safety_checker": None,
                "requires_safety_checker": False,
            }
            
            if is_local:
                kwargs["local_files_only"] = True
                print(f"[{device}] Loading OFFLINE from Kaggle Input: {model_id}")
            else:
                # Only use HF_TOKEN if loading remotely from Hugging Face
                hf_token_val = globals().get("HF_TOKEN", None)
                if hf_token_val:
                    kwargs["token"] = hf_token_val
                print(f"[{device}] Attempting remote download from Hugging Face: {model_id}")

            pipe = StableDiffusionPipeline.from_pretrained(model_id, **kwargs)
            print(f"[{device}] Successfully loaded {model_id}")
            break
        except Exception as e:
            print(f"[{device}] Could not load {model_id}: {e}")
            
    if pipe is None:
        raise RuntimeError(
            "No candidate model could be loaded. If you uploaded RoentGen-v2 as a Kaggle dataset, "
            "make sure the dataset is attached/mounted to this notebook session on the right-hand panel."
        )

    pipe = pipe.to(device)
    pipe.set_progress_bar_config(disable=True)

    # ---- VRAM optimizations (required: T4 has 16GB, 512x512 SD is heavy) ---
    pipe.enable_attention_slicing()          # required by spec
    pipe.vae.enable_slicing()                # extra headroom, decode-side
    try:
        pipe.enable_xformers_memory_efficient_attention()
    except Exception:
        pass  # xformers not installed / not available — attention slicing still applies

    pipe.unet.eval()
    pipe.vae.eval()
    pipe.text_encoder.eval()
    for p in pipe.unet.parameters():
        p.requires_grad_(False)
    for p in pipe.vae.parameters():
        p.requires_grad_(False)
    for p in pipe.text_encoder.parameters():
        p.requires_grad_(False)

    return pipe
print("Loading pipeline(s)...")
pipes = {d: load_pipeline(d) for d in DEVICES}
schedulers = {d: DDIMScheduler.from_config(pipes[d].scheduler.config) for d in DEVICES}
inverse_schedulers = {d: DDIMInverseScheduler.from_config(pipes[d].scheduler.config) for d in DEVICES}

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Scanning Kaggle Inputs for local model weights...
 -> Found local model at: /kaggle/input/datasets/pgc17ms072/roentgen-v2-complete
Resolved model search order: ['/kaggle/input/datasets/pgc17ms072/roentgen-v2-complete', 'stanfordmimi/RoentGen-v2', 'runwayml/stable-diffusion-v1-5']
Detected devices: ['cuda:0', 'cuda:1']
Loading pipeline(s)...
[cuda:0] Loading OFFLINE from Kaggle Input: /kaggle/input/datasets/pgc17ms072/roentgen-v2-complete


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

[cuda:0] Successfully loaded /kaggle/input/datasets/pgc17ms072/roentgen-v2-complete
[cuda:1] Loading OFFLINE from Kaggle Input: /kaggle/input/datasets/pgc17ms072/roentgen-v2-complete


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

[cuda:1] Successfully loaded /kaggle/input/datasets/pgc17ms072/roentgen-v2-complete


In [3]:
# %% [CELL 3] --------------------------------------------------------------
# Core building blocks: data loading, encoding, text embedding
# ---------------------------------------------------------------------------
@torch.no_grad()
def load_image_tensor_from_h5(h5_path: Path, image_id: str) -> torch.Tensor:
    """Loads an already-normalized [-1,1] float32 (512,512) grayscale array
    from the earlier preprocessing pipeline's HDF5 output, and replicates
    it to 3 channels (SD's VAE expects RGB-shaped input)."""
    with h5py.File(h5_path, "r") as h5f:
        if image_id not in h5f:
            raise KeyError(f"{image_id} not found in {h5_path}")
        arr = h5f[image_id][:]  # (512, 512), float32, already in [-1, 1]
    tensor = torch.from_numpy(arr).unsqueeze(0).unsqueeze(0)   # (1,1,H,W)
    tensor = tensor.repeat(1, 3, 1, 1)                          # (1,3,H,W)
    return tensor
 
 
@torch.no_grad()
def encode_to_latent(pipe: StableDiffusionPipeline, image_tensor: torch.Tensor) -> torch.Tensor:
    """VAE-encodes a real image into latent space. Uses the posterior MEAN
    (not a stochastic sample) so the starting point for inversion is
    deterministic — inversion accuracy depends on this being reproducible."""
    image_tensor = image_tensor.to(device=pipe.device, dtype=pipe.unet.dtype)
    latent_dist = pipe.vae.encode(image_tensor).latent_dist
    latents = latent_dist.mean * pipe.vae.config.scaling_factor
    return latents
 
 
@torch.no_grad()
def get_text_embeddings(pipe: StableDiffusionPipeline, prompt: str) -> torch.Tensor:
    """Returns a single (unconditional-scale) text embedding.
    NOTE: guidance_scale is fixed at 1.0 throughout this script. DDIM
    inversion is only mathematically well-defined when the exact same
    noise-prediction function is used forwards and backwards. Classifier-
    free guidance (scale > 1) mixes two different noise predictions
    (conditional + unconditional), which breaks that symmetry and makes
    inversion inexact. We therefore use plain conditional guidance at
    scale 1.0 (no CFG) for both inversion and ensemble generation.
    """
    tokens = pipe.tokenizer(
        prompt, padding="max_length", max_length=pipe.tokenizer.model_max_length,
        truncation=True, return_tensors="pt",
    ).input_ids.to(pipe.device)
    embeddings = pipe.text_encoder(tokens)[0].to(dtype=pipe.unet.dtype)
    return embeddings
 

In [4]:
# %% [CELL 4] --------------------------------------------------------------
# DDIM Inversion (deterministic, backwards: image -> z_T)
# ---------------------------------------------------------------------------
@torch.no_grad()
def ddim_invert(
    latents: torch.Tensor,
    pipe: StableDiffusionPipeline,
    inverse_scheduler: DDIMInverseScheduler,
    text_embeddings: torch.Tensor,
    num_inversion_steps: int = NUM_INVERSION_STEPS,
) -> torch.Tensor:
    """Deterministically inverts a clean latent z_0 into its corresponding
    noise latent z_T by running the DDIM update rule in reverse (increasing
    noise at each step instead of denoising). This recovers the EXACT
    starting point that would reconstruct the real image under standard
    DDIM sampling — critical for epistemic uncertainty, since we want to
    measure how the *model* varies around a fixed real starting point, not
    conflate that with variance from a random starting point.
    """
    inverse_scheduler.set_timesteps(num_inversion_steps, device=latents.device)
    latents = latents.clone()
 
    for t in inverse_scheduler.timesteps:
        noise_pred = pipe.unet(latents, t, encoder_hidden_states=text_embeddings).sample
        latents = inverse_scheduler.step(noise_pred, t, latents).prev_sample
 
    return latents  # this is z_T
 
 
# %% [CELL 5] --------------------------------------------------------------
# Stochastic DDIM sampling from a FIXED z_T (ensemble generation)
# ---------------------------------------------------------------------------
@torch.no_grad()
def ddim_sample_from_latent(
    z_T: torch.Tensor,
    pipe: StableDiffusionPipeline,
    scheduler: DDIMScheduler,
    text_embeddings: torch.Tensor,
    num_inference_steps: int = NUM_INFERENCE_STEPS,
    eta: float = ETA,
    generator: torch.Generator = None,
) -> torch.Tensor:
    """Runs the standard forward (denoising) DDIM loop starting from a FIXED
    z_T. With eta > 0, DDIM's step() injects fresh Gaussian noise scaled by
    the DDIM sigma_t at each step, using `generator` for reproducible-but-
    distinct draws — this is what makes each of the 20 ensemble runs differ
    despite sharing the same starting latent."""
    scheduler.set_timesteps(num_inference_steps, device=z_T.device)
    latents = z_T.clone()
 
    for t in scheduler.timesteps:
        noise_pred = pipe.unet(latents, t, encoder_hidden_states=text_embeddings).sample
        latents = scheduler.step(
            noise_pred, t, latents, eta=eta, generator=generator
        ).prev_sample
 
    return latents
 
 
@torch.no_grad()
def decode_latents_to_grayscale(pipe: StableDiffusionPipeline, latents: torch.Tensor) -> torch.Tensor:
    """VAE-decodes a latent back to pixel space, rescales [-1,1]->[0,1],
    and collapses the 3 (replicated) channels to a single grayscale plane
    matching the source CXR modality."""
    latents = latents / pipe.vae.config.scaling_factor
    image = pipe.vae.decode(latents).sample
    image = (image / 2 + 0.5).clamp(0, 1)
    image = image.float().cpu()
    gray = image.mean(dim=1).squeeze(0)  # (H, W)
    return gray
 
 
# %% [CELL 6] --------------------------------------------------------------
# Welford's Online Algorithm — O(1) memory pixel-wise mean/variance
# ---------------------------------------------------------------------------
class WelfordAccumulator:
    """Numerically-stable running mean/variance over a stream of tensors,
    without ever holding more than the current sample + two accumulators
    in memory. Kept on CPU in float32 to avoid competing with GPU VRAM."""
 
    def __init__(self, shape, device: str = "cpu"):
        self.n = 0
        self.mean = torch.zeros(shape, dtype=torch.float32, device=device)
        self.M2 = torch.zeros(shape, dtype=torch.float32, device=device)
 
    def update(self, x: torch.Tensor):
        x = x.to(dtype=torch.float32, device=self.mean.device)
        self.n += 1
        delta = x - self.mean
        self.mean += delta / self.n
        delta2 = x - self.mean
        self.M2 += delta * delta2
 
    @property
    def variance(self) -> torch.Tensor:
        if self.n < 2:
            return torch.zeros_like(self.mean)
        return self.M2 / (self.n - 1)   # sample variance (Bessel-corrected)
 
 
@torch.no_grad()
def run_uncertainty_ensemble(
    pipe: StableDiffusionPipeline,
    scheduler: DDIMScheduler,
    z_T: torch.Tensor,
    text_embeddings: torch.Tensor,
    num_ensemble: int = NUM_ENSEMBLE,
    num_inference_steps: int = NUM_INFERENCE_STEPS,
    eta: float = ETA,
    seed_base: int = 0,
):
    """Runs N stochastic reverse-diffusion passes from the same z_T,
    updating a Welford accumulator after each one. Each generated image
    is explicitly deleted and the CUDA cache cleared immediately after
    being folded into the running statistics — at no point do we hold
    more than 1 ensemble member on the GPU."""
    accumulator = WelfordAccumulator(shape=(IMG_SIZE, IMG_SIZE), device="cpu")
 
    for i in range(num_ensemble):
        generator = torch.Generator(device=pipe.device).manual_seed(seed_base + i)
 
        latents_i = ddim_sample_from_latent(
            z_T, pipe, scheduler, text_embeddings,
            num_inference_steps=num_inference_steps, eta=eta, generator=generator,
        )
        image_i = decode_latents_to_grayscale(pipe, latents_i)
 
        accumulator.update(image_i)
 
        # Strict cleanup — required for repeated 512x512 passes on a T4
        del latents_i, image_i, generator
        torch.cuda.empty_cache()
 
    return accumulator.mean, accumulator.variance
 
 
# %% [CELL 7] --------------------------------------------------------------
# Per-image and per-chunk processing
# ---------------------------------------------------------------------------
@torch.no_grad()
def process_one_image(
    pipe, scheduler, inverse_scheduler,
    image_id: str, finding: str, device: str,
) -> dict:
    prompt = f"a chest x-ray showing {finding.lower()}"
 
    image_tensor = load_image_tensor_from_h5(H5_PATH, image_id)
    text_embeddings = get_text_embeddings(pipe, prompt)
 
    latents_0 = encode_to_latent(pipe, image_tensor)
    z_T = ddim_invert(latents_0, pipe, inverse_scheduler, text_embeddings,
                       num_inversion_steps=NUM_INVERSION_STEPS)
 
    mean_img, var_img = run_uncertainty_ensemble(
        pipe, scheduler, z_T, text_embeddings,
        num_ensemble=NUM_ENSEMBLE, num_inference_steps=NUM_INFERENCE_STEPS, eta=ETA,
    )

    # Save the exact starting latent z_T — this is the SAME fixed point every
    # one of the 20 ensemble members was generated from, so persisting it
    # lets you re-run/extend the ensemble later, or verify reproducibility,
    # without re-doing DDIM inversion from scratch.
    z_T_saved = z_T.detach().to(dtype=torch.float16).cpu().numpy()

    result = {
        "mean": mean_img.numpy().astype(np.float32),
        "variance": var_img.numpy().astype(np.float32),
        "z_T": z_T_saved,
        "finding": finding,
    }

    del image_tensor, text_embeddings, latents_0, z_T, mean_img, var_img
    torch.cuda.empty_cache()
    gc.collect()
    return result
 
 
def process_partition(pipe, scheduler, inverse_scheduler, partition_df, device: str) -> dict:
    """Sequentially processes every row assigned to ONE device — strict
    batch size of 1 is enforced simply by never batching latents together;
    each call to process_one_image handles exactly one image end-to-end."""
    out = {}
    for _, row in tqdm(partition_df.iterrows(), total=len(partition_df),
                        desc=f"[{device}]", leave=False):
        image_id = row["image_id"]
        finding = row["finding"]
        try:
            out[image_id] = process_one_image(
                pipe, scheduler, inverse_scheduler, image_id, finding, device
            )
        except Exception as e:
            print(f"[{device}] FAILED on {image_id}: {e}")
    return out
 
 
def process_chunk(chunk_df, chunk_idx: int) -> dict:
    """Splits a 50-image chunk across all available GPUs and runs each
    partition concurrently via threads (CUDA calls release the GIL, so
    this achieves genuine cross-GPU overlap on a T4 x2 Kaggle instance)."""
    n_devices = len(DEVICES)
    partitions = np.array_split(chunk_df, n_devices)
 
    chunk_results = {}
    with ThreadPoolExecutor(max_workers=n_devices) as executor:
        futures = {
            executor.submit(
                process_partition,
                pipes[DEVICES[i]], schedulers[DEVICES[i]], inverse_schedulers[DEVICES[i]],
                partitions[i], DEVICES[i],
            ): DEVICES[i]
            for i in range(n_devices) if len(partitions[i]) > 0
        }
        for future in as_completed(futures):
            device = futures[future]
            try:
                chunk_results.update(future.result())
            except Exception as e:
                print(f"[{device}] partition failed entirely: {e}")
 
    return chunk_results
 
 
def save_chunk_npz(chunk_results: dict, chunk_idx: int, out_dir: Path = OUT_DIR):
    save_dict = {}
    meta_rows = []
    for image_id, data in chunk_results.items():
        save_dict[f"{image_id}__mean"] = data["mean"]
        save_dict[f"{image_id}__variance"] = data["variance"]
        save_dict[f"{image_id}__z_T"] = data["z_T"]   # fixed inversion starting point, fp16
        meta_rows.append({
            "image_id": image_id,
            "finding": data["finding"],
            "variance_mean": float(data["variance"].mean()),
            "variance_max": float(data["variance"].max()),
            "z_T_shape": str(data["z_T"].shape),
        })
 
    out_path = out_dir / f"uncertainty_chunk_{chunk_idx:03d}.npz"
    np.savez_compressed(out_path, **save_dict)
 
    meta_path = out_dir / f"uncertainty_chunk_{chunk_idx:03d}_meta.csv"
    pd.DataFrame(meta_rows).to_csv(meta_path, index=False)
 
    size_mb = out_path.stat().st_size / 1e6
    print(f"  Saved chunk {chunk_idx}: {len(chunk_results)} images -> "
          f"{out_path.name} ({size_mb:.1f} MB)")
 
 
# %% [CELL 8] --------------------------------------------------------------
# Build the rare-finding target list from the metadata CSV
# ---------------------------------------------------------------------------
metadata_df = pd.read_csv(METADATA_CSV)
 
# 'labels' was serialized as a Python-list string by the earlier pipeline
# (e.g. "['Pneumothorax']") — parse it back safely.
metadata_df["labels_parsed"] = metadata_df["labels"].apply(ast.literal_eval)
 
def first_matching_rare_label(labels):
    for lab in labels:
        if lab in RARE_FINDINGS:
            return lab
    return None
 
metadata_df["finding"] = metadata_df["labels_parsed"].apply(first_matching_rare_label)
target_df = metadata_df[metadata_df["finding"].notna()].reset_index(drop=True)
target_df = target_df[["image_id", "finding"]]
 
print(f"Targeting {len(target_df):,} images across findings: {RARE_FINDINGS}")
print(target_df["finding"].value_counts())
 
if len(target_df) == 0:
    raise RuntimeError(
        "0 images matched RARE_FINDINGS — check the exact class_name "
        "spelling in your CSV (annotations.class_name.unique()) and "
        "update RARE_FINDINGS accordingly."
    )
 
chunks = [target_df.iloc[i:i + CHUNK_SIZE] for i in range(0, len(target_df), CHUNK_SIZE)]
print(f"Split into {len(chunks)} chunks of up to {CHUNK_SIZE} images each.")
 

Targeting 138 images across findings: ['Atelectasis']
finding
Atelectasis    138
Name: count, dtype: int64
Split into 3 chunks of up to 50 images each.


In [6]:
'''import types
SMOKE_STEPS = 3
_orig_invert = ddim_invert
_orig_sample = ddim_sample_from_latent

def ddim_invert_fast(latents, pipe, sched, emb, num_inversion_steps=SMOKE_STEPS):
    return _orig_invert(latents, pipe, sched, emb, num_inversion_steps=SMOKE_STEPS)

def ddim_sample_fast(z_T, pipe, sched, emb, num_inference_steps=SMOKE_STEPS, eta=ETA, generator=None):
    return _orig_sample(z_T, pipe, sched, emb, num_inference_steps=SMOKE_STEPS, eta=eta, generator=generator)

ddim_invert = ddim_invert_fast
ddim_sample_from_latent = ddim_sample_fast
NUM_ENSEMBLE_BACKUP, NUM_ENSEMBLE = NUM_ENSEMBLE, 3   # shrink ensemble too

print(f"Smoke test image_id={test_row['image_id']} finding={test_row['finding']}")
result = process_one_image(pipes[device], schedulers[device], inverse_schedulers[device],
                            test_row["image_id"], test_row["finding"], device)
print(f"Smoke test OK in {time.time()-t0:.1f}s")
print("mean shape:", result["mean"].shape, "variance shape:", result["variance"].shape)
print("variance range:", result["variance"].min(), "-", result["variance"].max())

# restore real functions/settings before moving to Step 4
ddim_invert = _orig_invert
ddim_sample_from_latent = _orig_sample
NUM_ENSEMBLE = NUM_ENSEMBLE_BACKUP
# %% [DIAGNOSTIC — smoke test, ignore timing here]
SMOKE_STEPS = 3
test_row = target_df.iloc[0]
device = DEVICES[0]

t0 = time.time()
result = process_one_image(
    pipes[device], schedulers[device], inverse_schedulers[device],
    image_id=test_row["image_id"], finding=test_row["finding"], device=device,
)'''

'import types\nSMOKE_STEPS = 3\n_orig_invert = ddim_invert\n_orig_sample = ddim_sample_from_latent\n\ndef ddim_invert_fast(latents, pipe, sched, emb, num_inversion_steps=SMOKE_STEPS):\n    return _orig_invert(latents, pipe, sched, emb, num_inversion_steps=SMOKE_STEPS)\n\ndef ddim_sample_fast(z_T, pipe, sched, emb, num_inference_steps=SMOKE_STEPS, eta=ETA, generator=None):\n    return _orig_sample(z_T, pipe, sched, emb, num_inference_steps=SMOKE_STEPS, eta=eta, generator=generator)\n\nddim_invert = ddim_invert_fast\nddim_sample_from_latent = ddim_sample_fast\nNUM_ENSEMBLE_BACKUP, NUM_ENSEMBLE = NUM_ENSEMBLE, 3   # shrink ensemble too\n\nprint(f"Smoke test image_id={test_row[\'image_id\']} finding={test_row[\'finding\']}")\nresult = process_one_image(pipes[device], schedulers[device], inverse_schedulers[device],\n                            test_row["image_id"], test_row["finding"], device)\nprint(f"Smoke test OK in {time.time()-t0:.1f}s")\nprint("mean shape:", result["mean"].shape, "var

In [7]:
'''# %% [DIAGNOSTIC — real settings, warm vs steady-state]
device = DEVICES[0]
torch.cuda.reset_peak_memory_stats(device)

row_a = target_df.iloc[1]
row_b = target_df.iloc[2]

t0 = time.time()
_ = process_one_image(pipes[device], schedulers[device], inverse_schedulers[device],
                       row_a["image_id"], row_a["finding"], device)
warm_elapsed = time.time() - t0
print(f"Image 1 (includes CUDA warmup): {warm_elapsed:.1f}s")

t0 = time.time()
result = process_one_image(pipes[device], schedulers[device], inverse_schedulers[device],
                            row_b["image_id"], row_b["finding"], device)
steady_elapsed = time.time() - t0
print(f"Image 2 (steady-state): {steady_elapsed:.1f}s")

peak_vram_gb = torch.cuda.max_memory_allocated(device) / 1e9
print(f"Peak VRAM on {device}: {peak_vram_gb:.2f} GB")'''

'# %% [DIAGNOSTIC — real settings, warm vs steady-state]\ndevice = DEVICES[0]\ntorch.cuda.reset_peak_memory_stats(device)\n\nrow_a = target_df.iloc[1]\nrow_b = target_df.iloc[2]\n\nt0 = time.time()\n_ = process_one_image(pipes[device], schedulers[device], inverse_schedulers[device],\n                       row_a["image_id"], row_a["finding"], device)\nwarm_elapsed = time.time() - t0\nprint(f"Image 1 (includes CUDA warmup): {warm_elapsed:.1f}s")\n\nt0 = time.time()\nresult = process_one_image(pipes[device], schedulers[device], inverse_schedulers[device],\n                            row_b["image_id"], row_b["finding"], device)\nsteady_elapsed = time.time() - t0\nprint(f"Image 2 (steady-state): {steady_elapsed:.1f}s")\n\npeak_vram_gb = torch.cuda.max_memory_allocated(device) / 1e9\nprint(f"Peak VRAM on {device}: {peak_vram_gb:.2f} GB")'

In [8]:
'''# %% [DIAGNOSTIC — mini chunk, both GPUs]
mini_chunk = target_df.iloc[3:7].reset_index(drop=True)   # 4 fresh images, 2 per GPU
t0 = time.time()
mini_results = process_chunk(mini_chunk, chunk_idx=999)
elapsed = time.time() - t0

print(f"4 images across {len(DEVICES)} GPUs: {elapsed:.1f}s wall-clock")
print(f"Reference: 1 image on 1 GPU (Step 4) = 72.4s")
print(f"If perfectly parallel, 4 images / 2 GPUs should take ~2x72.4 = {2*72.4:.1f}s")
print(f"Succeeded: {len(mini_results)}/4")'''

'# %% [DIAGNOSTIC — mini chunk, both GPUs]\nmini_chunk = target_df.iloc[3:7].reset_index(drop=True)   # 4 fresh images, 2 per GPU\nt0 = time.time()\nmini_results = process_chunk(mini_chunk, chunk_idx=999)\nelapsed = time.time() - t0\n\nprint(f"4 images across {len(DEVICES)} GPUs: {elapsed:.1f}s wall-clock")\nprint(f"Reference: 1 image on 1 GPU (Step 4) = 72.4s")\nprint(f"If perfectly parallel, 4 images / 2 GPUs should take ~2x72.4 = {2*72.4:.1f}s")\nprint(f"Succeeded: {len(mini_results)}/4")'

In [9]:
'''# %% [DIAGNOSTIC — extrapolate total runtime]
per_image_per_gpu = 154.5 / (4 / len(DEVICES))   # effective steady-state cost per image, parallelized
total_images = len(target_df)
est_total_seconds = (total_images / len(DEVICES)) * per_image_per_gpu

print(f"Effective per-image cost (parallelized): {per_image_per_gpu:.1f}s")
print(f"Total target images: {total_images}")
print(f"Estimated total runtime: {est_total_seconds/3600:.2f} hours "
      f"across {len(DEVICES)} GPUs")
print(f"Estimated chunks: {len(chunks)}, "
      f"~{(CHUNK_SIZE/len(DEVICES))*per_image_per_gpu/60:.1f} min/chunk")'''

'# %% [DIAGNOSTIC — extrapolate total runtime]\nper_image_per_gpu = 154.5 / (4 / len(DEVICES))   # effective steady-state cost per image, parallelized\ntotal_images = len(target_df)\nest_total_seconds = (total_images / len(DEVICES)) * per_image_per_gpu\n\nprint(f"Effective per-image cost (parallelized): {per_image_per_gpu:.1f}s")\nprint(f"Total target images: {total_images}")\nprint(f"Estimated total runtime: {est_total_seconds/3600:.2f} hours "\n      f"across {len(DEVICES)} GPUs")\nprint(f"Estimated chunks: {len(chunks)}, "\n      f"~{(CHUNK_SIZE/len(DEVICES))*per_image_per_gpu/60:.1f} min/chunk")'

In [10]:
'''sample_id = list(mini_results.keys())[0]
var = mini_results[sample_id]["variance"]
mean = mini_results[sample_id]["mean"]
print("mean range:", mean.min(), mean.max())
print("variance range:", var.min(), var.max())
print("variance is all-zero:", np.allclose(var, 0))'''

'sample_id = list(mini_results.keys())[0]\nvar = mini_results[sample_id]["variance"]\nmean = mini_results[sample_id]["mean"]\nprint("mean range:", mean.min(), mean.max())\nprint("variance range:", var.min(), var.max())\nprint("variance is all-zero:", np.allclose(var, 0))'

In [11]:
# %% [CELL 9] --------------------------------------------------------------
# Main loop: process every chunk, saving variance heatmaps after each
# ---------------------------------------------------------------------------
for chunk_idx, chunk_df in enumerate(chunks):
    t0 = time.time()
    print(f"\n=== Chunk {chunk_idx + 1}/{len(chunks)} ({len(chunk_df)} images) ===")
 
    chunk_results = process_chunk(chunk_df, chunk_idx)
    save_chunk_npz(chunk_results, chunk_idx)
 
    elapsed = time.time() - t0
    print(f"  Chunk {chunk_idx} done in {elapsed / 60:.1f} min "
          f"({elapsed / max(len(chunk_results), 1):.1f} s/image)")
 
    del chunk_results
    gc.collect()
    for d in DEVICES:
        torch.cuda.empty_cache()
 
print("\nAll chunks complete. Variance heatmaps saved to /kaggle/working/ "
      "as uncertainty_chunk_*.npz — publish as a Kaggle Dataset to persist.")
 


=== Chunk 1/12 (50 images) ===


/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


[cuda:0]:   0%|          | 0/25 [00:00<?, ?it/s]

[cuda:1]:   0%|          | 0/25 [00:00<?, ?it/s]

  Saved chunk 0: 50 images -> uncertainty_chunk_000.npz (90.4 MB)
  Chunk 0 done in 34.2 min (41.1 s/image)

=== Chunk 2/12 (50 images) ===


...

[cuda:1]:   0%|          | 0/25 [00:00<?, ?it/s]

  Saved chunk 10: 50 images -> uncertainty_chunk_010.npz (90.5 MB)
  Chunk 10 done in 33.8 min (40.5 s/image)

=== Chunk 12/12 (28 images) ===


[cuda:1]:   0%|          | 0/14 [00:00<?, ?it/s]

[cuda:0]:   0%|          | 0/14 [00:00<?, ?it/s]

  Saved chunk 11: 28 images -> uncertainty_chunk_011.npz (50.6 MB)
  Chunk 11 done in 19.0 min (40.6 s/image)

All chunks complete. Variance heatmaps saved to /kaggle/working/ as uncertainty_chunk_*.npz — publish as a Kaggle Dataset to persist.
